<a href="https://colab.research.google.com/github/KhadijaAL15/discount-strategy-analysis/blob/main/01_data_cleaning_and_merging_discount_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Business Questions

In [ ]:
# How should products be classified into different categories to simplify reports and analysis?
# What is the distribution of product prices across different categories?
# How many products are being discounted?
# How big are the offered discounts as a percentage of the product prices?
# How do seasonality and special dates (Christmas, Black Friday) affect sales? - more sales for sure
# How could data collection be improved?

# DataFrames

In [ ]:
# DataFrame url

orderlines_url = "https://drive.google.com/file/d/15X8ePvpvdq3FxRmFpt-kiwbtQR6jS6Z_/view?usp=drive_link"
orders_url = "https://drive.google.com/file/d/1Nf1BVxoBx-ujv57O3mgJM8ExP1RTcFg-/view?usp=drive_link"
products_url = "https://drive.google.com/file/d/1qxoq95v9SAPngurb8jzleysjnR0b3rr-/view?usp=drive_link"
brands_url = "https://drive.google.com/file/d/15MXV0WY7awEUTP2L7XnNtz-bstZ17aC0/view?usp=drive_link"

In [ ]:
# import Url-Function

def Format_url(url):

    path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
    return pd.read_csv(path)

# Data Frame Cleaning

## Products

In [ ]:
# products_df

products_df = Format_url(products_url)
products_df_copy = products_df.copy()
products_df_copy

,sku,name,desc,price,promo_price,in_stock,type
0,RAI0007,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,59.99,499.899,1,8696
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59,589.996,0,13855401
2,APP0025,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,59,569.898,0,1387
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,25,229.997,0,1230
4,KIN0007,Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM,2GB RAM Mac mini and iMac (2006/07) MacBook Pr...,34.99,31.99,1,1364
...,...,...,...,...,...,...,...
19321,BEL0376,Belkin Travel Support Apple Watch Black,compact and portable stand vertically or horiz...,29.99,269.903,1,12282
19322,THU0060,"Enroute Thule 14L Backpack MacBook 13 ""Black",Backpack with capacity of 14 liter compartment...,69.95,649.903,1,1392
19323,THU0061,"Enroute Thule 14L Backpack MacBook 13 ""Blue",Backpack with capacity of 14 liter compartment...,69.95,649.903,1,1392
19324,THU0062,"Enroute Thule 14L Backpack MacBook 13 ""Red",Backpack with capacity of 14 liter compartment...,69.95,649.903,0,1392


In [ ]:
products_df_copy.shape

(19326, 7)

In [ ]:
products_df_copy.duplicated().sum()

np.int64(8746)

In [ ]:
products_df_copy[['price', 'promo_price']].head(20)

#too many different values

In [ ]:
(products_df_copy['promo_price'] > products_df_copy['price']).sum()

In [ ]:
# Boolean mask to find the orders that contain a price with multiple decimal points
multiple_decimal_mask = products_df_copy['promo_price'].str.count(r"\.") > 1

# Apply the boolean mask to the orderlines DataFrame. To find the 'order_id' of all the affected orders.
corrupted_order_ids = products_df_copy.loc[multiple_decimal_mask, "sku"]

# Keep only the rows that do not have multiple decimal points
products_df_copy = prodcuts_df_copy.loc[~products_df_copy['sku'].isin(corrupted_order_ids)]

In [ ]:
# produts_df.shape[0]
# more than 60% data removed- not acceptable. drop whole 'promo_price' column
# The discounts are reflected in orderlines_df 'unit_price'

In [ ]:
products_df_copy["price"] = pd.to_numeric(products_df_copy["price"])

In [ ]:
# drop all with no unique sku

products_unique = products_df_copy.drop_duplicates(subset=['sku'])


In [ ]:
df_products_copy.isna().any()

## Orders

In [ ]:
# orders_df
orders_df = Format_url(orders_url)
orders_df_copy = orders_df.copy()

# Create copy of DataFrame
orders_df_copy.info()

# total_paid column has 5 missing values
# created_date not correct Datatype

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      226909 non-null  int64  
 1   created_date  226909 non-null  object 
 2   total_paid    226904 non-null  float64
 3   state         226909 non-null  object 
dtypes: float64(1), int64(1), object(2)
memory usage: 6.9+ MB


In [ ]:
orders_df_copy.duplicated().sum()
# 0 duplicates in orders_df_copy

np.int64(0)

In [ ]:
print(f"5 missing values represent {((orders_df_copy.total_paid.isna().sum() / orders_df_copy.shape[0])*100).round(5)}% of the rows in our DataFrame")

5 missing values represent 0.0022% of the rows in our DataFrame


In [ ]:
# Deletion of missing values as they only make up 0,0022% of the orders_df_copy
orders_df_copy = orders_df_copy.loc[~orders_df_copy.total_paid.isna(), :]

In [ ]:
# Change Datatype for created_date to datetime
orders_df_copy["created_date"] = pd.to_datetime(orders_df_copy["created_date"])

In [ ]:
orders_df_copy.tail(50)

,order_id,created_date,total_paid,state
226859,527352,2018-03-14 13:29:25,37.97,Shopping Basket
226860,527353,2018-03-14 13:33:06,17.98,Pending
226861,527354,2018-03-14 13:38:08,99.95,Pending
226862,527355,2018-03-14 13:38:43,27.97,Pending
226863,527356,2018-03-14 13:31:14,17.98,Place Order
226864,527357,2018-03-14 13:31:38,13.99,Place Order
226865,527358,2018-03-14 13:31:45,31.97,Place Order
226866,527359,2018-03-14 13:33:07,9.99,Shopping Basket
226867,527360,2018-03-14 13:35:32,83.94,Pending
226868,527361,2018-03-14 13:34:11,13.99,Shopping Basket


In [ ]:
# Filter 'state' columns = completed
orders_completed = orders_df_copy[orders_df_copy['state'] == 'Completed']


In [ ]:
# New df with only state completed
orders_completed

In [ ]:
orders_df_copy['state']

,state
1,Completed
2,Completed
3,Completed
5,Completed
6,Completed
...,...
226549,Completed
226577,Completed
226581,Completed
226603,Completed


## Orderlines

In [ ]:
# orderlines_df
orderlines_df = Format_url(orderlines_url)
orderlines_df.head(44)

,id,id_order,product_id,product_quantity,sku,unit_price,date
0,1119109,299539,0,1,OTT0133,18.99,2017-01-01 00:07:19
1,1119110,299540,0,1,LGE0043,399.00,2017-01-01 00:19:45
2,1119111,299541,0,1,PAR0071,474.05,2017-01-01 00:20:57
3,1119112,299542,0,1,WDT0315,68.39,2017-01-01 00:51:40
4,1119113,299543,0,1,JBL0104,23.74,2017-01-01 01:06:38
5,1119114,295310,0,10,WDT0249,231.79,2017-01-01 01:14:27
6,1119115,299544,0,1,APP1582,1.137.99,2017-01-01 01:17:21
7,1119116,299545,0,1,OWC0100,47.49,2017-01-01 01:46:16
8,1119119,299546,0,1,IOT0014,18.99,2017-01-01 01:50:34
9,1119120,295347,0,1,APP0700,72.19,2017-01-01 01:54:11


In [ ]:
# Copy orderlines_df
orderlines_df_copy = orderlines_df.copy()

In [ ]:
orderlines_df_copy.info()

# DataType 'created_date' -> Date Time, 'unit_price' -> Float

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 293983 entries, 0 to 293982
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   id                293983 non-null  int64 
 1   id_order          293983 non-null  int64 
 2   product_id        293983 non-null  int64 
 3   product_quantity  293983 non-null  int64 
 4   sku               293983 non-null  object
 5   unit_price        293983 non-null  object
 6   date              293983 non-null  object
dtypes: int64(4), object(3)
memory usage: 15.7+ MB


In [ ]:
# Count duplicates
orderlines_df_copy.duplicated().sum()

# orderlines_df_copy has no duplicates

np.int64(0)

In [ ]:
# Count missing values
missing_values = orderlines_df_copy.isna().sum()
print(f'{missing_values}')

# orderlines_df_copy has no missing values

id                  0
id_order            0
product_id          0
product_quantity    0
sku                 0
unit_price          0
date                0
dtype: int64


In [ ]:
# Correct the DataTypes
orderlines_df_copy['date'] = pd.to_datetime(orderlines_df_copy['date'])

# Changes value of date from object to datetime.
orderlines_df_copy['unit_price'].head(50)

# Column 'unit_price' cant change DataType to Float.
orderlines_df_copy['unit_price'] = pd.to_numeric(orderlines_df_copy['unit_price'])


# Assumption: Values with 6 digits have 2 decimal points. Should set a "." on index [-2]

,unit_price
0,18.99
1,399.00
2,474.05
3,68.39
4,23.74
5,231.79
6,1.137.99
7,47.49
8,18.99
9,72.19


In [ ]:
# Show rows that have an 'id_order' in orderlines_df that DOESN'T appear in 'orders_df'
missing_orders = orderlines_df_copy.loc[~orderlines_df_copy['id_order'].isin(orders_df_copy['order_id'])]

In [ ]:
# Show rows with missing values
print(missing_orders.isnull().sum())

id                  0
id_order            0
product_id          0
product_quantity    0
sku                 0
unit_price          0
date                0
dtype: int64


In [ ]:
# Show rows that have an "id_order" with AT LEAST one missing 'unit_price'
orders_with_missing_prices = missing_orders[missing_orders['unit_price'].isnull()]['id_order'].unique()

In [ ]:
# Count orderlines that contain an 'id_order' from 'orders_with_missing_prices' above
affected_rows = orderlines_df_copy[orderlines_df_copy['id_order'].isin(orders_with_missing_prices)]
print(f"Number of orderlines to remove: {affected_rows.shape[0]}")

# Output the percentage of total:
percent = (affected_rows.shape[0] / orderlines_df_copy.shape[0]) * 100
print(f"Percentage of total: {percent:.2f}%")

Number of orderlines to remove: 0
Percentage of total: 0.00%


In [ ]:
# Remove all orderlines from "affected_rows" above
clean_orderlines = orderlines_df_copy[~orderlines_df_copy['id_order'].isin(orders_with_missing_prices)]

In [ ]:
# How many rows are left after removing "affected_rows"?
remaining_missing_orders = missing_orders[~missing_orders['id_order'].isin(orders_with_missing_prices)]
print(f"Remaining missing_orders rows: {remaining_missing_orders.shape[0]}")

Remaining missing_orders rows: 240


In [ ]:
# Those orderlines are being excluded from the analysis because they reference orders that do not exist in the 'orders_df'.
# Including such entries could result in analyzing products sold without a valid, completed transaction, which would not accurately reflect actual company revenue or customer activity.
final_orderlines = clean_orderlines[clean_orderlines['id_order'].isin(orders_df_copy['order_id'])]

In [ ]:
# To correct the 'unit_price' column- check for Formatting errors.
# Checks which values have more than one "."
final_orderlines.loc[(final_orderlines["unit_price"].str.count(r"\.") > 1)].count()


,0
id,36137
id_order,36137
product_id,36137
product_quantity,36137
sku,36137
unit_price,36137
date,36137


In [ ]:
# Checks if values have no "."
final_orderlines.loc[~final_orderlines['unit_price'].str.contains(r'\.', na=False)]

# All values contain either one or two dots.

,id,id_order,product_id,product_quantity,sku,unit_price,date


In [ ]:
print(final_orderlines['unit_price'].dtype)
print(final_orderlines['unit_price'].head())

object
0     18.99
1    399.00
2    474.05
3     68.39
4     23.74
Name: unit_price, dtype: object


In [ ]:
# Remove the decimal points of all digits and set "." at index number [-2]
fix_prices = final_orderlines['unit_price'] = final_orderlines['unit_price'].str.replace(".","", regex = False).apply(lambda x: x[:-2]+ "." + x[-2:])
final_orderlines['unit_price'].head(20)

# Filters all the "." and removes them from the string, with apply(lamda) we apply at[-2]a dot.

/tmp/ipykernel_4394/3030740978.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fix_prices = final_orderlines['unit_price'] = final_orderlines['unit_price'].str.replace(".","", regex = False).apply(lambda x: x[:-2]+ "." + x[-2:])


,unit_price
0,18.99
1,399.00
2,474.05
3,68.39
4,23.74
6,1137.99
7,47.49
8,18.99
9,72.19
10,35.14


In [ ]:
# Search for missing values or nulls after replacing the dot
print(final_orderlines['unit_price'].isnull().sum())

# There are 0 null values in final_orderlines -> 'unit_price'

0


In [ ]:
# Corrected the decimal point of 'unit_price'
# We can now change the DataType to numeric
final_orderlines['unit_price']= pd.to_numeric(final_orderlines['unit_price'])


In [ ]:
# Print values which equals or a smaller than 0
zero_values = final_orderlines[final_orderlines['unit_price'] <= 0].count()
print(zero_values)


# There are 866 rows which equals or is smaller than 0.
# Since the 'unit_price' have a negative value or equals zero we will drop these rows.


id                  865
id_order            865
product_id          865
product_quantity    865
sku                 865
unit_price          865
date                865
dtype: int64


In [ ]:
# Identify the unique 'SKU' of the rows to be dropped
rows_to_drop = final_orderlines[final_orderlines['unit_price'] <= 0][['sku', 'unit_price']]
rows_to_drop

,sku,unit_price
53515,KIN0153-2,0.0
53530,WDT0347,0.0
56529,LIBRO,0.0
56562,LIBRO,0.0
56576,LIBRO,0.0
...,...,...
291880,SYN0150,0.0
292216,APP0699,0.0
292388,WAC0251,0.0
292905,OWC0110,0.0


In [ ]:
# Drops the 866 rows with 0 or negative. values
zero_values = final_orderlines[final_orderlines['unit_price'] <= 0].index
final_orderlines = final_orderlines.drop(zero_values)

In [ ]:
# Print values which are higher than 10000
big_values = final_orderlines[final_orderlines['unit_price'] > 10000]
print(big_values)

# 33 Prices with values above 10000, needs to be decided wethere to drop these rows or fix these values

             id  id_order  product_id  product_quantity      sku  unit_price  \
36979   1197439    331780           0                 1  NEA0009   159989.83   
38317   1200059    332976           0                 6  LAC0223    15349.00   
61757   1241776    352790           0                 3  LAC0223    14099.00   
63244   1244301    354084           0                 4  LAC0223    14099.00   
106679  1321964    391352           0                 1  LAC0223    10012.80   
122028  1361930    404678           0                 1  LAC0223    10006.80   
232052  1551007    487164           0                 1  APP2660    14580.00   
232077  1551059    487192           0                 1  APP2660    14580.00   
232754  1551977    487490           0                 1  APP2660    12175.24   
234834  1555558    488935           0                 4  APP2660    14580.00   
235157  1556015    489083           0                 1  APP2694    12071.00   
241744  1567362    390450           0   

In [ ]:
# Identify wether these prices are realtistic or not.
# Compare to Products_df
products_df_copy[products_df_copy["sku"] == "NEA0009"]

# Products with 'SKU' 'NEA0009' - 'unit_price' in orderlines_df doesnt match with 'price' in Products_df.
# Since this information is corrupted we will drop this row.
# Other Prices are realistic since these are prices for laptops that have CUSTOM hardware.


,sku,name,desc,price,promo_price,in_stock,type
1025,NEA0009,Netatmo home thermostat for iPhone and iPad,Thermostat iPhone iPad wifi Mac design Stark.,179,174.989,0,11905404


In [ ]:
# Drops rows with 'SKU' 'NEA0009'
corrupted_price = final_orderlines[final_orderlines['sku']== "NEA0009"].index
final_orderlines = final_orderlines.drop(corrupted_price)

In [ ]:
final_orderlines[final_orderlines["sku"] == "NEA0009"]

,id,id_order,product_id,product_quantity,sku,unit_price,date


In [ ]:
# Check which prices = 0,01 cent and identify wether to remove or keep the values
final_orderlines[final_orderlines['unit_price'] == 0.01]


,id,id_order,product_id,product_quantity,sku,unit_price,date
68348,1253375,358440,0,1,SEV0018,0.01,2017-05-23 16:21:58
71325,1258848,359758,0,1,SEV0021,0.01,2017-05-31 13:31:24
96326,1303336,382458,0,1,SEV0041,0.01,2017-07-26 17:44:43


In [ ]:
# Compare the Sku´s to the products_df
skus_to_remove = products_df_copy.loc[products_df_copy['sku'].isin(['SEV0018','SEV0021','SEV0041']), 'sku']

# All 3 rows are Installation prices and not products prices therefore we will drop these 3 rows.

In [ ]:
# Drop 3 rows
final_orderlines = final_orderlines[~final_orderlines['sku'].isin(skus_to_remove)]


In [ ]:
# Check the description of the values
final_orderlines['unit_price'].describe().round(2)

# Description shows false values.
# Values need to be corrected above.
# 866 rows =< 0 dropped, 1 > 10000 dropped, 3 rows == 0.01 = 870 rows dropped
# From 293983 rows, 870 rows haven been dropped = 0,295 % lost Data inseatd of 12,3% and 35169 rows

,unit_price
count,292064.00
mean,411.29
std,785.38
min,0.29
25%,35.99
50%,92.99
75%,343.99
max,15349.00


In [ ]:
final_orderlines.info()

<class 'pandas.core.frame.DataFrame'>
Index: 292064 entries, 0 to 293982
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   id                292064 non-null  int64         
 1   id_order          292064 non-null  int64         
 2   product_id        292064 non-null  int64         
 3   product_quantity  292064 non-null  int64         
 4   sku               292064 non-null  object        
 5   unit_price        292064 non-null  float64       
 6   date              292064 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(4), object(1)
memory usage: 17.8+ MB


In [ ]:
# Identify 'completed' orders that have no corresponding orderlines_df
orders_without_orderlines = orders_completed[~orders_completed['order_id'].isin(final_orderlines['id_order'])]

In [ ]:
# Create a unique list of IDs for orders that need to be removed
orders_to_remove = orders_without_orderlines['order_id'].unique()

In [ ]:
# Remove the identified orders without orderlines from the final_orderlines_df
orders_completed = orders_completed[~orders_completed['order_id'].isin(orders_to_remove)]

In [ ]:
# Cleaned orderlines_df = final_orderlines
final_orderlines

## Brands


In [ ]:
# brands_df

brands_df = Format_url(brands_url)
brands_df_copy = brands_df.copy()
brands_df_copy

,short,long
0,8MO,8Mobility
1,ACM,Acme
2,ADN,Adonit
3,AII,Aiino
4,AKI,Akitio
...,...,...
182,XOO,Xoopar
183,XRI,X-Rite
184,XTO,Xtorm
185,ZAG,ZaggKeys


In [ ]:
# Show Info

brands_df_copy.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 187 entries, 0 to 186
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   short   187 non-null    object
 1   long    187 non-null    object
dtypes: object(2)
memory usage: 3.1+ KB


In [ ]:
# count duplicates

brands_df_copy.duplicated().sum()

# No duplicates found
# No missing values

np.int64(0)

# Merging of DataFrames

In [ ]:
# Define uniques in orders_completed
valid_ids = orders_completed['order_id'].unique()

In [ ]:
# # Filter final_orderlines to keep only those belonging to valid order
orderlines_filtered = final_orderlines[final_orderlines['id_order'].isin(valid_ids)]

In [ ]:
# Identify orderlines that belong to orders no longer in the main dataset
lost_orderlines = final_orderlines[~final_orderlines['id_order'].isin(valid_ids)]
lost_orderlines.head(50)


,id,id_order,product_id,product_quantity,sku,unit_price,date
0,1119109,299539,0,1,OTT0133,18.99,2017-01-01 00:07:19
1,1119110,299540,0,1,LGE0043,399.00,2017-01-01 00:19:45
2,1119111,299541,0,1,PAR0071,474.05,2017-01-01 00:20:57
3,1119112,299542,0,1,WDT0315,68.39,2017-01-01 00:51:40
4,1119113,299543,0,1,JBL0104,23.74,2017-01-01 01:06:38
6,1119115,299544,0,1,APP1582,1137.99,2017-01-01 01:17:21
10,1119125,299548,0,5,SPE0132,35.14,2017-01-01 02:02:20
12,1119127,299550,0,1,WDT0309,149.14,2017-01-01 02:09:52
13,1119129,299552,0,1,MBI0010,16.99,2017-01-01 02:13:51
14,1119130,299552,0,1,SAT0044,19.99,2017-01-01 02:14:36


In [ ]:
print(lost_orderlines.describe())

                 id       id_order  product_id  product_quantity  \
count  2.301650e+05  230165.000000    230165.0     230165.000000   
mean   1.401126e+06  421538.279291         0.0          1.121256   
min    1.119109e+06  241319.000000         0.0          1.000000   
25%    1.262186e+06  362289.000000         0.0          1.000000   
50%    1.417032e+06  428911.000000         0.0          1.000000   
75%    1.537944e+06  482102.000000         0.0          1.000000   
max    1.650203e+06  527401.000000         0.0        999.000000   
std    1.534574e+05   66403.000544         0.0          3.816655   

          unit_price                           date  
count  230165.000000                         230165  
mean      457.261943  2017-09-21 02:23:21.780926720  
min         0.290000            2017-01-01 00:07:19  
25%        39.990000            2017-06-06 02:48:13  
50%       104.640000            2017-11-20 19:38:46  
75%       386.000000            2018-01-04 16:57:11  
max     1

In [ ]:
# Merge orders_completed with final_orderlines
merged_order_orderlines = orders_completed.merge(final_orderlines,
                                                left_on="order_id",
                                                right_on="id_order",
                                                how="inner")

In [ ]:
# Final merged DataFrame
merged_order_orderlines

,order_id,created_date,total_paid,state,id,id_order,product_id,product_quantity,sku,unit_price,date
0,241423,2017-11-06 13:10:02,136.15,Completed,1398738,241423,0,1,LAC0212,129.16,2017-11-06 12:47:20
1,242832,2017-12-31 17:40:03,15.76,Completed,1529178,242832,0,1,PAR0074,10.77,2017-12-31 17:26:40
2,243330,2017-02-16 10:59:38,84.98,Completed,1181923,243330,0,1,OWC0074,77.99,2017-02-15 17:07:44
3,245275,2017-06-28 11:35:37,149.00,Completed,1276706,245275,0,1,TAD0007,149.00,2017-06-28 11:12:30
4,245595,2017-01-21 12:52:47,112.97,Completed,1154394,245595,0,2,PAC1561,52.99,2017-01-21 12:49:00
...,...,...,...,...,...,...,...,...,...,...,...
61894,527042,2018-03-14 11:47:50,18.98,Completed,1649446,527042,0,1,APP0927,13.99,2018-03-14 11:42:38
61895,527070,2018-03-14 11:50:48,24.97,Completed,1649512,527070,0,2,APP0698,9.99,2018-03-14 11:49:01
61896,527074,2018-03-14 11:51:42,24.97,Completed,1649522,527074,0,2,APP0698,9.99,2018-03-14 11:49:36
61897,527096,2018-03-14 11:58:40,34.96,Completed,1649565,527096,0,3,APP0698,9.99,2018-03-14 11:54:35


In [ ]:
merged_order_orderlines.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61899 entries, 0 to 61898
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          61899 non-null  int64         
 1   created_date      61899 non-null  datetime64[ns]
 2   total_paid        61899 non-null  float64       
 3   state             61899 non-null  object        
 4   id                61899 non-null  int64         
 5   id_order          61899 non-null  int64         
 6   product_id        61899 non-null  int64         
 7   product_quantity  61899 non-null  int64         
 8   sku               61899 non-null  object        
 9   unit_price        61899 non-null  float64       
 10  date              61899 non-null  datetime64[ns]
dtypes: datetime64[ns](2), float64(2), int64(5), object(2)
memory usage: 5.2+ MB


In [ ]:
merged_order_orderlines.describe()

,order_id,created_date,total_paid,id,id_order,product_id,product_quantity,unit_price,date
count,61899.000000,61899,61899.000000,6.189900e+04,61899.000000,61899.0,61899.000000,61899.000000,61899
mean,414484.815086,2017-09-12 02:40:11.066802688,405.779409,1.385919e+06,414484.815086,0.0,1.121456,240.357070,2017-09-11 20:55:48.443739136
min,241423.000000,2017-01-01 01:51:47,0.000000,1.119116e+06,241423.000000,0.0,1.000000,0.290000,2017-01-01 01:46:16
25%,362848.000000,2017-06-12 12:47:29.500000,56.980000,1.266212e+06,362848.000000,0.0,1.000000,28.990000,2017-06-12 10:30:11.500000
50%,417154.000000,2017-10-26 20:33:34,139.000000,1.389436e+06,417154.000000,0.0,1.000000,69.990000,2017-10-26 13:53:57
75%,470121.000000,2017-12-25 00:35:45,390.330000,1.513792e+06,470121.000000,0.0,1.000000,186.790000,2017-12-24 19:37:20.500000
max,527112.000000,2018-03-14 12:03:52,13387.780000,1.649593e+06,527112.000000,0.0,72.000000,8287.800000,2018-03-14 11:58:13
std,65457.207701,NaN,732.936017,1.505847e+05,65457.207701,0.0,0.777325,502.704137,NaN


# CSV Files




In [ ]:
# Convert merged_order_orderlines into csv file
merged_order_orderlines.to_csv('merged_total_cleaned.csv', index=False)

In [ ]:
# Convert orderlines_df into csv file
final_orderlines.to_csv('cleaned_orderlines.csv', index=False)

In [ ]:
# Convert orders_df into csv file
orders_completed.to_csv('cleaned_orders.csv', index=False)